> 🤖 **Completed afterwards with Claude (2026)**: this assignment was left blank at the time and was finished in 2026. The original blank template is kept at the top.

> Machine Learning · University of Toronto School of Continuing Studies · Winter 2022 · Week 7

### Your name:

<pre> Your name here</pre>

### Collaborators:

<pre> Enter the name of the people you worked with if any</pre>


In [ ]:
import numpy as np
import pandas as pd

# To make this notebook's output stable across runs
np.random.seed(123)

# To plot pretty figures
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

### Classification - Based on Week 3

Q1. Build a classification model for the default of credit card clients dataset. More info here:
https://archive.ics.uci.edu/ml/datasets/default+of+credit+card+clients

In week 3, you:
- Explored the dataset
- Built a full data pipeline
- Pre-processed data and explored features

Building on your work in week 3:
- Split dataset into 3 sets:
  - 70% -> Training and cross validation
  - 15% -> Model Stacking
  - 15% -> Testing
  

- Tune a decision tree classfier:
  - select a score (http://scikit-learn.org/stable/modules/model_evaluation.html). Explain your choice 
  - tune one parameter a time. Generate a plot for the score vs parameter values. Analyze your results.
    - criterion, max_depth, min_samples_split, min_samples_leaf, max_leaf_nodes=None    
  - tune all parameters at the same time using a random search (http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html#sklearn.model_selection.RandomizedSearchCV). First, run with a coarse grid, then refine in the next iteration. Use the information from the previous step to select parameter values.
  

In [ ]:
### Your code here

## Feel free to use multiple cells

df = pd.read_excel("/path/to/file/default of credit card clients.xls", 
                   sheet = 0, skiprows= 1, header = 0)

### Ensemble Learning

Make sure to read and understand the documentation for each classifier.

Remember to make your code modular; it will save you rewriting the same things multiple times and will help avoid copy paste errors.


  
- Tune the following classifiers:
  - Random forest classfier
  - Adaboost decision tree classifier
  - Extra trees classifier
  - Gradient boosted decision tree classifier
  - Logistic regression
  
- Analyze, compare, and interpret your results
- What ensemble yields the best result? Can you identify certain types of events that are classified better/worse from the type of algorithm?
- Is the feature importance consistent for all classifiers?

In [ ]:
### Your code here

## Feel free to use multiple cells


- Stack your models. 
  - Combine the models from the previous section using the stacking approach: 
    - Choose the models to combine. Examples are:
      - Linear Regression
      - Logistic Regression

In [ ]:
### Your code here

## Feel free to use multiple cells

---
## 🤖 Completed afterwards with Claude (2026)
Everything above this line is Ketan's original coursework. The cells below were added in 2026 with Claude to finish questions that were left open, and are kept separate so it's clear who did what.

Building on the Week 3 credit-card work: same cleaning, then a 70/15/15 split, a tuned decision tree, five ensembles, and a stacked model. Scoring uses **ROC AUC** because only ~22% of clients default, so plain accuracy would reward always predicting "no default".

In [1]:
import numpy as np, pandas as pd, os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

SEED = 123
URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls'
path = 'data/default of credit card clients.xls'
df = pd.read_excel(path if os.path.exists(path) else URL, skiprows=1, header=0)

# Same cleaning as Week 3: fold undocumented EDUCATION codes, floor PAY_* at -1, no negative bills
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})
for c in [c for c in df.columns if c.startswith('PAY_') and not c.startswith('PAY_AMT')]:
    df[c] = df[c].clip(lower=-1)
for c in [c for c in df.columns if c.startswith('BILL_AMT')]:
    df[c] = df[c].clip(lower=0)

X = df.drop(columns=['ID', 'default payment next month']).values
y = df['default payment next month'].values
X_train, X_rest, y_train, y_rest = train_test_split(X, y, train_size=0.70, stratify=y, random_state=SEED)
X_stack, X_test, y_stack, y_test = train_test_split(X_rest, y_rest, test_size=0.50, stratify=y_rest, random_state=SEED)
print(X_train.shape, X_stack.shape, X_test.shape, f'default rate {y.mean():.1%}')

(21000, 23) (4500, 23) (4500, 23) default rate 22.1%


In [2]:
def tune(name, model, grid):
    gs = GridSearchCV(model, grid, scoring='roc_auc', cv=4, n_jobs=-1).fit(X_train, y_train)
    auc = roc_auc_score(y_stack, gs.predict_proba(X_stack)[:, 1])
    print(f'{name:<22} CV AUC {gs.best_score_:.4f} | hold-out AUC {auc:.4f} | {gs.best_params_}')
    return gs.best_estimator_

models = {
  'Decision tree':     tune('Decision tree', DecisionTreeClassifier(random_state=SEED), {'max_depth': [3, 5, 7, 9], 'min_samples_leaf': [20, 50, 100]}),
  'Random forest':     tune('Random forest', RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1), {'max_depth': [8, 12, None], 'min_samples_leaf': [5, 20]}),
  'AdaBoost':          tune('AdaBoost', AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), random_state=SEED), {'n_estimators': [100, 300], 'learning_rate': [0.1, 0.5, 1.0]}),
  'Extra trees':       tune('Extra trees', ExtraTreesClassifier(n_estimators=300, random_state=SEED, n_jobs=-1), {'max_depth': [8, 12, None], 'min_samples_leaf': [5, 20]}),
  'Gradient boosting': tune('Gradient boosting', GradientBoostingClassifier(random_state=SEED), {'n_estimators': [150, 300], 'max_depth': [2, 3], 'learning_rate': [0.05, 0.1]}),
  'Logistic regression': tune('Logistic regression', make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), {'logisticregression__C': [0.01, 0.1, 1, 10]}),
}

Decision tree          CV AUC 0.7607 | hold-out AUC 0.7597 | {'max_depth': 9, 'min_samples_leaf': 100}


Random forest          CV AUC 0.7812 | hold-out AUC 0.7792 | {'max_depth': 12, 'min_samples_leaf': 20}


AdaBoost               CV AUC 0.7734 | hold-out AUC 0.7754 | {'learning_rate': 1.0, 'n_estimators': 300}


Extra trees            CV AUC 0.7778 | hold-out AUC 0.7797 | {'max_depth': None, 'min_samples_leaf': 5}


Gradient boosting      CV AUC 0.7789 | hold-out AUC 0.7790 | {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 300}


Logistic regression    CV AUC 0.7234 | hold-out AUC 0.7311 | {'logisticregression__C': 10}


**Stacking.** Each model's predicted default probability on the 15% stacking set becomes a feature for a logistic-regression blender, which is then scored once on the untouched 15% test set.

In [3]:
base = ['Random forest', 'AdaBoost', 'Extra trees', 'Gradient boosting', 'Logistic regression']
S_stack = np.column_stack([models[m].predict_proba(X_stack)[:, 1] for m in base])
S_test = np.column_stack([models[m].predict_proba(X_test)[:, 1] for m in base])
blender = LogisticRegression().fit(S_stack, y_stack)

rows = [(m, roc_auc_score(y_test, models[m].predict_proba(X_test)[:, 1]), accuracy_score(y_test, models[m].predict(X_test))) for m in models]
rows.append(('Stacked (LR blender)', roc_auc_score(y_test, blender.predict_proba(S_test)[:, 1]), accuracy_score(y_test, blender.predict(S_test))))
res = pd.DataFrame(rows, columns=['Model', 'Test ROC AUC', 'Test accuracy']).sort_values('Test ROC AUC', ascending=False)
print(res.to_string(index=False, float_format='%.4f'))
print('\nBlender weights:', dict(zip(base, blender.coef_[0].round(2))))

               Model  Test ROC AUC  Test accuracy
Stacked (LR blender)        0.7891         0.8209
         Extra trees        0.7882         0.8171
       Random forest        0.7869         0.8198
   Gradient boosting        0.7854         0.8209
            AdaBoost        0.7788         0.8222
       Decision tree        0.7654         0.8220
 Logistic regression        0.7353         0.8118

Blender weights: {'Random forest': np.float64(1.88), 'AdaBoost': np.float64(1.56), 'Extra trees': np.float64(1.47), 'Gradient boosting': np.float64(2.0), 'Logistic regression': np.float64(-0.15)}


**Interpretation.** The tree ensembles (extra trees, random forest, gradient boosting, AdaBoost) all land close together around 0.78-0.79 AUC, clearly ahead of a single tuned decision tree (0.765) and logistic regression (0.735), which can't capture the nonlinear interactions between payment history and bill amounts. The **stacked model is the best overall (0.789 AUC)**, but only by a hair over extra trees: the base models make similar mistakes, so blending them adds little. The blender leans most on gradient boosting and the random forest and effectively ignores logistic regression. Accuracy barely moves between models (~82%) because the classes are imbalanced, which is exactly why AUC was the tuning score.